# Curious Account Creation

1. Load file of IDs and passwords.
2. Invite each ID to applet.
3. Create an account for each ID using password from file.
4. As each account, log in and accept invitation.

In [ ]:
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
import httpx
import notebookutils
import polars as pl

from curious_pyapi.api.curious import CuriousApiClient
from curious_pyapi.exceptions.exceptions import allow_existing
from curious_pyapi.schema.curious import CuriousId
from curious_pyapi.schema.pyapi import CuriousAuth
from curious_pyapi.utils.logging import get_logger

conf = notebookutils.variableLibrary.getLibrary("ConfigurationVariables")

LOGGER = get_logger(__name__)

APPLET_ID = conf.getVariable("AppletIdProlificMentalStateModelingStudy")
EMAIL_FORMAT = "msm+Z{}@childmind.org"
TEMPORARY_NAME_FORMAT = "_{}"


def map_prolific_to_curious(df: pl.DataFrame, pl_namespace: str) -> pl.DataFrame:
    """Transform a username-password DataFrame into a CuriousAccount DataFrame."""
    record_id_str = pl.col("record_id").cast(pl.String).str.zfill(6)
    match pl_namespace:
        case "new_curious_user":
            _name_expr = pl.format(TEMPORARY_NAME_FORMAT, record_id_str)
        case _:
            _name_expr = pl.lit("")

    return getattr(
        df.with_columns(
            email=pl.format(EMAIL_FORMAT, record_id_str),
            password=pl.col("password").cast(pl.String),
            firstName=_name_expr,
            lastName=_name_expr,
            language=pl.lit("en"),
            secretUserId=pl.col("record_id").cast(pl.String),
            nickname=pl.col("record_id").cast(pl.String),
            tag=pl.lit("Prolific"),
        ),
        pl_namespace,
    ).enforce_schema()


def create_accounts_from_df(
    client: CuriousApiClient,
    df: pl.DataFrame,
    applet_id: CuriousId,
    user_type: str = "respondent",
    *,
    limit=None,
) -> None:
    """For each row in DataFrame, create an account in Curious."""
    invitations: dict[str, CuriousId] = {}
    api_df = map_prolific_to_curious(df, "curious_account")
    if limit:
        api_df = api_df.head(limit)
    for record in api_df.to_dicts():
        invitation = allow_existing(
            client.invite_user(applet_id=applet_id, user_type=user_type).post,
            json=record,
        )
        if invitation:
            invitation = invitation.json().get("result")
        if not invitation:
            all_invitations = (
                client.invitations()
                .get(params={"appletId": applet_id, "userType": "respondent"})
                .json()
                .get("result", [])
            )
            this_user_invitations = [
                i
                for i in all_invitations
                if all(
                    [
                        i.get("appletId") == applet_id,
                        i.get("secretUserId") == record["secretUserId"],
                    ]
                )
            ]
            if this_user_invitations:
                invitation = this_user_invitations[0]
        if invitation and "key" in invitation:
            invitations[record["secretUserId"]] = invitation["key"]
    new_user_df = map_prolific_to_curious(df, "new_curious_user")
    for user in new_user_df.head(limit).to_dicts():
        response = allow_existing(client.user_create.post, json=user)
        secret_id = user["firstName"][1:]
        if secret_id in invitations:
            user_client = CuriousApiClient(
                auth=CuriousAuth(
                    curious_email=user["email"], curious_password=user["password"]
                )
            )
            response = user_client.invitation_accept(key=invitations[secret_id]).post()
            if not httpx.codes.is_success(response.status_code):
                response.raise_for_status()
            del user_client

In [ ]:
secret_client = SecretClient(
    vault_url=conf.getVariable("KeyVaultUri"),
    credential=DefaultAzureCredential(),
)
admin_credentials = CuriousAuth(
    curious_email=secret_client.get_secret("CuriousEmail").value,
    curious_password=secret_client.get_secret("CuriousPassword").value,
    applet_id=APPLET_ID,
    applet_password=secret_client.get_secret("MsmProlificAppletPassword").value,
)

## 1. Load file of IDs and passwords

In [ ]:
df = pl.read_excel(Path("Files/study_passwords.xlsx"))
"""Raw DataFrame of IDs and passwords."""

## 2‒4
### 2. Invite each ID to applet
### 3. Create an account for each ID using password from file.
### 4. As each account, log in and accept invitation.

In [ ]:
admin_client = CuriousApiClient(auth=admin_credentials)
create_accounts_from_df(admin_client, df, applet_id=APPLET_ID)